In [5]:
#list of herbal and fruit;
#Tannin, Mangoesteen peel, Curcumin, Moringa Oleifera leaves, Quercetin, rosmarinic acid, Chinese gingseng, Korean ginseng 248101 (4.24% w/v ginsenoside content), Korean ginseng 246302 (4.81% w/v ginsenoside content), Korean ginseng 248102 (6.15% w/v ginsenoside content).

#Installing necessary system tools and Python libraries;

- Tesseract-OCR: Used for Optical Character Recognition to read text from images.
- Libraries: Installs pytesseract for text extraction and BaselineRemoval for post-processing the spectrum.
- Imports: Loads essential data science and image processing libraries, including cv2 (OpenCV), numpy, pandas, scipy, and plotly.

In [6]:
!sudo apt install tesseract-ocr #เครื่องมือสำหรับอ่านข้อความจากภาพ (Optical Character Recognition).
!pip install pytesseract BaselineRemoval

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [7]:
import cv2 #OpenCV สำหรับจัดการรูปภาพ
import numpy as np #สำหรับจัดการข้อมูลตัวเลข
import pandas as pd #สำหรับจัดการข้อมูลตัวเลข
import plotly.graph_objects as go #แสดงผลกราฟ
import matplotlib.pyplot as plt # Import matplotlib.pyplot for plotting
import plotly.express as px
from scipy.signal import find_peaks #identify peak position
from plotly.subplots import make_subplots
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
from BaselineRemoval import BaselineRemoval

# Tannin Extraction

In [8]:
#------------ Raman spectrum of tannin extract ----------------
#need to reverse scale

# 1. Load Image
img_path = '/content/tanin.png'
img = cv2.imread(img_path)

# Convert BGR to RGB for matplotlib display
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 2. Advanced Color Masking for Brownish Spectrum Line
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
# Defining a range for dark reds/browns to isolate the trace
lower_red = np.array([0, 50, 20])
upper_red = np.array([20, 255, 150])
mask = cv2.inRange(hsv, lower_red, upper_red) #เพื่อเก็บเฉพาะพิกเซลที่เป็นเส้นกราฟไว้

# 3. Extract Coordinates
y_coords, x_coords = np.where(mask > 0)

if len(x_coords) == 0:
    print("Error: No spectrum line detected. Please check the image path or color thresholds.")
else:
    # Use pandas to clean the noisy line (take the median Y for every X)
    df_pixels = pd.DataFrame({'x': x_coords, 'y': y_coords})
    df_pixels = df_pixels.groupby('x').median().reset_index()

    x_pix = df_pixels['x'].values
    y_pix = df_pixels['y'].values

    # 4. Physical Axis Mapping (Tanin image range: 3100 to 100 cm⁻¹), as show on original tannin figure
    x_min_pix, x_max_pix = x_pix.min(), x_pix.max()
    y_baseline_pix = np.max(y_coords)

    def pixel_to_unit(x, x_min, x_max, val_start, val_end):
        # Maps pixels to wavenumbers accounting for the reversed scale (3100 -> 100)
        return val_start - (x - x_min) * (val_start - val_end) / (x_max - x_min)

    wavenumbers_raw = pixel_to_unit(x_pix, x_min_pix, x_max_pix, 3100, 100)
    intensities_raw = (y_baseline_pix - y_pix)

    # 5. Standardization (450 - 1800 cm⁻¹ with 1 cm⁻¹ step)
    standard_x = np.arange(450, 1801, 1)

    ## Sort raw data for interpolation since the original image scale was reversed
    sort_idx = np.argsort(wavenumbers_raw)
    f = interp1d(wavenumbers_raw[sort_idx], intensities_raw[sort_idx],
                 bounds_error=False, fill_value="extrapolate")
    interp_y = f(standard_x)

    # 6. Preprocessing
    # Smoothing using Savitzky-Golay (window_length=31 for noise suppression)
    # Savitzky-Golay เพื่อลดรอยหยักของเส้นกราฟ.
    y_smoothed = savgol_filter(interp_y, window_length=31, polyorder=3)

    # Baseline Removal using IModPoly
    #IModPoly เพื่อดึงเส้นฐานของกราฟที่เอียงให้กลับมาตรง.
    base_obj = BaselineRemoval(y_smoothed)
    y_corrected = base_obj.IModPoly(degree=2)

    # Min-Max Normalization (0.0 to 1.0)
    y_norm = (y_corrected - np.min(y_corrected)) / (np.max(y_corrected) - np.min(y_corrected))

    # 7. Interactive Visualization (Plotly)
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Original tannin Image", "tanin spectrum extraction"))

    fig.add_trace(px.imshow(img_rgb).data[0], row=1, col=1)

    fig.add_trace(go.Scatter(
        x=standard_x,
        y=y_norm,
        mode='lines',
        line=dict(color='blue', width=1.5),
        name='Intensity'
    ), row=1, col=2)

    fig.update_layout(template='plotly_white', hovermode='x unified')
    fig.show()

    # 8. Export Data
    df_final = pd.DataFrame({'Wavenumber': standard_x, 'Intensity': y_norm})
    df_final.to_csv('tanin_data.csv', index=False)
    print("Success: 'tanin_data.csv' has been generated.")

    df_to_process = df_final # Assign df_final AFTER it has been created

    peak_results = []

    # Identify columns that contain spectral data (excluding 'Wavenumber')
    intensity_cols = [col for col in df_to_process.columns if col != 'Wavenumber']

    for col in intensity_cols:
        # Find peaks: adjust 'height' or 'prominence' based on your specific data noise levels
        # prominence=0.01 is a good starting point for normalized data
        peaks, _ = find_peaks(df_to_process[col], prominence=0.01)

        for peak_idx in peaks:
            peak_results.append({
                'Wavenumber': df_to_process['Wavenumber'].iloc[peak_idx],
                'Intensity': df_to_process[col].iloc[peak_idx]
            })

    # Export peak position
    # Create a DataFrame from the results
    df_peaks = pd.DataFrame(peak_results)
    df_peaks.to_csv('peak_positions_tannin.csv', index=False)
    print("Peak identification complete. Results saved to 'peak_positions_tannin.csv'.")
    df_peaks.head()

Success: 'tanin_data.csv' has been generated.
Peak identification complete. Results saved to 'peak_positions_tannin.csv'.


# Mangoesteen Peel

In [9]:
#------------ Raman spectrum of mangoesteen peel, crude extract ----------------
#this picture show as a lot of noise, the result is try to reduce noise

# 1. Load Image
img_path = '/content/mangoesteen peel.png'
img = cv2.imread(img_path)

# Convert BGR to RGB for matplotlib display
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 2. Advanced Color Masking for Brownish Spectrum Line
# Mangosteen line is bright red; adjusted HSV to capture pure red tones
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
lower_red1 = np.array([0, 100, 100])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([160, 100, 100])
upper_red2 = np.array([180, 255, 255])

mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
mask = cv2.add(mask1, mask2) # Captures red across the HSV wrap-around

# 3. Extract Coordinates
y_coords, x_coords = np.where(mask > 0)

if len(x_coords) == 0:
    print("Error: No spectrum line detected. Please check the image path or color thresholds.")
else:
    # Use pandas to clean the noisy line (take the median Y for every X)
    df_pixels = pd.DataFrame({'x': x_coords, 'y': y_coords})
    df_pixels = df_pixels.groupby('x').median().reset_index()

    x_pix = df_pixels['x'].values
    y_pix = df_pixels['y'].values

    # 4. Physical Axis Mapping (Tanin image range: 3100 to 100 cm⁻¹)
    x_min_pix, x_max_pix = x_pix.min(), x_pix.max()
    y_baseline_pix = np.max(y_coords)

    def pixel_to_unit(x, x_min, x_max, val_start, val_end):
        return val_start - (x - x_min) * (val_start - val_end) / (x_max - x_min)

    wavenumbers_raw = pixel_to_unit(x_pix, x_min_pix, x_max_pix, 850, 1900)
    intensities_raw = (y_baseline_pix - y_pix)

    # 5. Standardization (450 - 1800 cm⁻¹ with 1 cm⁻¹ step)
    standard_x = np.arange(450, 1801, 1)

    ## Sort raw data for interpolation since the original image scale was reversed
    sort_idx = np.argsort(wavenumbers_raw)
    f = interp1d(wavenumbers_raw[sort_idx], intensities_raw[sort_idx],
                 bounds_error=False, fill_value="extrapolate")
    interp_y = f(standard_x)

    # 6. Preprocessing
    # Smoothing using Savitzky-Golay (window_length=31 for noise suppression)
    # Savitzky-Golay เพื่อลดรอยหยักของเส้นกราฟ.
    y_smoothed = savgol_filter(interp_y, window_length=31, polyorder=3)

    # Baseline Removal using IModPoly
    #IModPoly เพื่อดึงเส้นฐานของกราฟที่เอียงให้กลับมาตรง.
    base_obj = BaselineRemoval(y_smoothed)
    y_corrected = base_obj.IModPoly(degree=2)

    # Min-Max Normalization (0.0 to 1.0)
    y_norm = (y_corrected - np.min(y_corrected)) / (np.max(y_corrected) - np.min(y_corrected))

    # 7. Interactive Visualization (Plotly)
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Original mangoesteen peel Image", "mangoesteen peel spectrum extraction"))

    fig.add_trace(px.imshow(img_rgb).data[0], row=1, col=1)

    fig.add_trace(go.Scatter(
        x=standard_x,
        y=y_norm,
        mode='lines',
        line=dict(color='blue', width=1.5),
        name='Intensity'
    ), row=1, col=2)

    fig.update_layout(template='plotly_white', hovermode='x unified')
    fig.show()

    # 8. Export Data
    df_final = pd.DataFrame({'Wavenumber': standard_x, 'Intensity': y_norm})
    df_final.to_csv('mangoesteen peel_data.csv', index=False)
    print("Success: 'mangoesteen peel_data.csv' has been generated.")

    df_to_process = df_final # Assign df_final AFTER it has been created

    peak_results = []

    # Identify columns that contain spectral data (excluding 'Wavenumber')
    intensity_cols = [col for col in df_to_process.columns if col != 'Wavenumber']

    for col in intensity_cols:
        # Find peaks: adjust 'height' or 'prominence' based on your specific data noise levels
        # prominence=0.01 is a good starting point for normalized data
        peaks, _ = find_peaks(df_to_process[col], prominence=0.01)

        for peak_idx in peaks:
            peak_results.append({
                'Wavenumber': df_to_process['Wavenumber'].iloc[peak_idx],
                'Intensity': df_to_process[col].iloc[peak_idx]
            })

    # Export peak position
    # Create a DataFrame from the results
    df_peaks = pd.DataFrame(peak_results)
    df_peaks.to_csv('peak_positions_mangoesteen peel.csv', index=False)
    print("Peak identification complete. Results saved to 'peak_positions_mangoesteen peel.csv'.")
    df_peaks.head()

Success: 'mangoesteen peel_data.csv' has been generated.
Peak identification complete. Results saved to 'peak_positions_mangoesteen peel.csv'.


# Curcumin Extract

In [10]:
#------------ Raman spectrum of curcumin extract ----------------

# 1. Load Image
img_path = '/content/curcumin.png' # Ensure this matches your filename
img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 2. Pre-process to Isolate the Line
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
# Using a slightly higher threshold to ensure the thin line is captured
_, threshold = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

# 3. Use Contour Detection to remove labels (numbers)
# This finds all "shapes" in the mask
contours, _ = cv2.findContours(threshold, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Create a blank mask to draw only the longest contour (the spectrum line)
clean_mask = np.zeros_like(threshold)
if contours:
    # The spectrum line is usually the contour with the largest area or length
    spectrum_contour = max(contours, key=cv2.contourArea)
    cv2.drawContours(clean_mask, [spectrum_contour], -1, 255, thickness=cv2.FILLED)

# 4. Extract Coordinates from the cleaned mask
y_coords, x_coords = np.where(clean_mask > 0)

if len(x_coords) == 0:
    print("Error: Detection failed. Trying fallback...")
    # Fallback: manually crop the area above the line to avoid labels
    # (Assuming labels are always placed vertically above peaks)
else:
    df_pixels = pd.DataFrame({'x': x_coords, 'y': y_coords})
    # Take the minimum Y (highest point) for each X to get the peak profile
    df_pixels = df_pixels.groupby('x').min().reset_index()

    x_pix = df_pixels['x'].values
    y_pix = df_pixels['y'].values

    # 5. Physical Axis Mapping (600 to 1800 cm⁻¹)
    x_min_pix, x_max_pix = x_pix.min(), x_pix.max()
    y_baseline_pix = np.max(y_coords)

    def pixel_to_unit(x, x_min, x_max, val_start, val_end):
        return val_start + (x - x_min) * (val_end - val_start) / (x_max - x_min)

    #in the study of this curcumin sample, assign spectral range from 700 to 1900 cm-1
    #but when running the code, the spectrum is expanded from original.
    #this may due to the start of original spectrum is not actully start at 700 cm-1, as original curcumin provided
    #therefore, we set the range between 735 and 1800, as the spectrum appear
    wavenumbers_raw = pixel_to_unit(x_pix, x_min_pix, x_max_pix, 735, 1800)
    intensities_raw = (y_baseline_pix - y_pix)

    # 6. Standardization (Standard 1 cm⁻¹ step)
    standard_x = np.arange(450, 1801, 1)
    f = interp1d(wavenumbers_raw, intensities_raw, bounds_error=False, fill_value="extrapolate")
    interp_y = f(standard_x)

    # 7. Preprocessing
    # window_length must be odd. 11 is good for sharp peaks.
    y_smoothed = savgol_filter(interp_y, window_length=11, polyorder=3)

    # Baseline Removal
    base_obj = BaselineRemoval(y_smoothed)
    y_corrected = base_obj.IModPoly(degree=2)

    # Min-Max Normalization
    y_norm = (y_corrected - np.min(y_corrected)) / (np.max(y_corrected) - np.min(y_corrected))

    # 8. Interactive Comparison Plot
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Original curcumin Image", "curcumin spectrum extraction"))

    fig.add_trace(px.imshow(img_rgb).data[0], row=1, col=1)

    fig.add_trace(go.Scatter(
        x=standard_x,
        y=y_norm,
        mode='lines',
        line=dict(color='blue', width=1.5),
        name='Intensity'
    ), row=1, col=2)

    fig.update_layout(template='plotly_white', hovermode='x unified')
    fig.show()

    # 8. Export Data
    df_final = pd.DataFrame({'Wavenumber': standard_x, 'Intensity': y_norm})
    df_final.to_csv('curcumin_data.csv', index=False)
    print("Success: 'curcumin_data.csv' has been generated.")

    df_to_process = df_final # Assign df_final AFTER it has been created

    peak_results = []

    # Identify columns that contain spectral data (excluding 'Wavenumber')
    intensity_cols = [col for col in df_to_process.columns if col != 'Wavenumber']

    for col in intensity_cols:
        # Find peaks: adjust 'height' or 'prominence' based on your specific data noise levels
        # prominence=0.01 is a good starting point for normalized data
        peaks, _ = find_peaks(df_to_process[col], prominence=0.01)

        for peak_idx in peaks:
            peak_results.append({
                'Wavenumber': df_to_process['Wavenumber'].iloc[peak_idx],
                'Intensity': df_to_process[col].iloc[peak_idx]
            })

    # Export peak position
    # Create a DataFrame from the results
    df_peaks = pd.DataFrame(peak_results)
    df_peaks.to_csv('peak_positions_curcumin.csv', index=False)
    print("Peak identification complete. Results saved to 'peak_positions_curcumin.csv'.")
    df_peaks.head()

Success: 'curcumin_data.csv' has been generated.
Peak identification complete. Results saved to 'peak_positions_curcumin.csv'.


# Moringa Oleifera leaves

In [11]:
#------------ Raman spectrum of Leaves From Moringa Oleifera ----------------
#error at the begining of spectrum, might occur from the frame of original pic

# 1. Load Image
img_path = '/content/Leaves From Moringa Oleife_re.png' # Ensure this matches your filename
img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 2. Pre-process
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
# Clean up noise and isolate black lines
_, threshold = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY_INV)

# 3. Extract Coordinates
y_pixels, x_pixels = np.where(threshold > 0)
df_pixels = pd.DataFrame({'x': x_pixels, 'y': y_pixels})

# --- CRITICAL FIX 1: Filter Plot Area ---
# Exclude the very bottom (axis line) and very top (titles)
height, width = gray.shape
df_filtered = df_pixels[(df_pixels['y'] > height * 0.1) & (df_pixels['y'] < height * 0.85)]

# --- CRITICAL FIX 2: Select the correct line ---
# In image coords, the "top" of a peak is the LOWEST Y value.
# We take the minimum Y for each X to follow the upper profile of the spectrum.
df_spectrum = df_filtered.groupby('x')['y'].min().reset_index()

x_pix = df_spectrum['x'].values
y_pix = df_spectrum['y'].values

# 4. Physical Axis Mapping
# The image shows a range from roughly 400 to 3800 cm-1
x_min_pix, x_max_pix = x_pix.min(), x_pix.max()
# Use the filtered max Y as the baseline (bottom of the graph)
y_baseline_pix = df_filtered['y'].max()

def pixel_to_unit(x, x_min, x_max, val_start, val_end):
    return val_start + (x - x_min) * (val_end - val_start) / (x_max - x_min)

# Map to the full range visible in the original image
wavenumbers_raw = pixel_to_unit(x_pix, x_min_pix, x_max_pix, 280, 3850)
intensities_raw = (y_baseline_pix - y_pix)

# 5. Standardization
# Match the range of the original plot (approx 500 to 3700)
standard_x = np.arange(450, 1801, 1)
f = interp1d(wavenumbers_raw, intensities_raw, bounds_error=False, fill_value="extrapolate")
interp_y = f(standard_x)

# 6. Preprocessing
y_smoothed = savgol_filter(interp_y, window_length=11, polyorder=3)
base_obj = BaselineRemoval(y_smoothed)
y_corrected = base_obj.IModPoly(degree=2)

# Min-Max Normalization
y_norm = (y_corrected - np.min(y_corrected)) / (np.max(y_corrected) - np.min(y_corrected))

# 7. Interactive Comparison Plot
fig = make_subplots(rows=1, cols=2, subplot_titles=("Original Leaves From Moringa Oleifera Image", "spectrum Leaves From Moringa Oleifera extraction"))
fig.add_trace(px.imshow(img_rgb).data[0], row=1, col=1)

fig.add_trace(go.Scatter(
    x=standard_x,
    y=y_norm,
    mode='lines',
    line=dict(color='blue', width=1.5),
    name='Intensity'
), row=1, col=2)

fig.update_layout(template='plotly_white', hovermode='x unified')
fig.show()

# 8. Export
df_final = pd.DataFrame({'Wavenumber': standard_x, 'Intensity': y_norm})
df_final.to_csv('moringa_oleifera_data.csv', index=False)
print("Success: 'moringa_oleifera_data.csv' has been generated.")

df_to_process = df_final # Assign df_final AFTER it has been created

peak_results = []

# Identify columns that contain spectral data (excluding 'Wavenumber')
intensity_cols = [col for col in df_to_process.columns if col != 'Wavenumber']

for col in intensity_cols:
    # Find peaks: adjust 'height' or 'prominence' based on your specific data noise levels
    # prominence=0.01 is a good starting point for normalized data
    peaks, _ = find_peaks(df_to_process[col], prominence=0.01)

    for peak_idx in peaks:
        peak_results.append({
            'Wavenumber': df_to_process['Wavenumber'].iloc[peak_idx],
            'Intensity': df_to_process[col].iloc[peak_idx]
        })

# Export peak position
# Create a DataFrame from the results
df_peaks = pd.DataFrame(peak_results)
df_peaks.to_csv('peak_positions_moringa_oleifera.csv', index=False)
print("Peak identification complete. Results saved to 'peak_positions_moringa_oleifera.csv'.")
df_peaks.head()

Success: 'moringa_oleifera_data.csv' has been generated.
Peak identification complete. Results saved to 'peak_positions_moringa_oleifera.csv'.


,Wavenumber,Intensity
0,461,0.168887
1,497,0.109137
2,519,0.122703
3,547,0.148244
4,588,0.142871


# Quercetin

In [12]:
#------------ Raman spectrum of quercetin ----------------
#need to reverse scale

# 1. Load Image
img_path = '/content/quercetin_re.png'
img = cv2.imread(img_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 2. Advanced Pre-processing
# We use a very tight threshold to isolate the black line
# and then apply a slight blur to connect any small gaps in the line
_, binary = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY_INV)

# 3. Refined ROI (Cutting off the labels and axes strictly)
h, w = binary.shape
# We crop deeply from the top to remove "(a) quercetin..."
# and deeply from the bottom/sides to remove axis numbers
roi_y_start, roi_y_end = int(h*0.18), int(h*0.88)
roi_x_start, roi_x_end = int(w*0.06), int(w*0.96)
roi_mask = binary[roi_y_start:roi_y_end, roi_x_start:roi_x_end]

# 4. Extracting the "Bottom-most" pixel for the curve
# This helps ignore text floating ABOVE the curve
y_coords, x_coords = np.where(roi_mask > 0)
df_pixels = pd.DataFrame({'x': x_coords, 'y': y_coords})

# Key Fix: In the 1500-700 range, text is above the line.
# Taking the MAX y-coordinate (bottom of the black pixels)
# helps isolate the spectrum from the text labels.
df_pixels = df_pixels.groupby('x')['y'].max().reset_index()

# 5. Mapping to Wavenumber (Reverse Scale: ~1700 to 200)
x_min_pix, x_max_pix = df_pixels['x'].min(), df_pixels['x'].max()

# The original image spans roughly 1750 on the left to 200 on the right
def pixel_to_unit(x, x_min, x_max):
    return 1750 - (x - x_min) * (1550 / (x_max - x_min))

wavenumbers_raw = pixel_to_unit(df_pixels['x'].values, x_min_pix, x_max_pix)
# Invert Y (0 is top in image pixels, we want high peaks to be 'up')
intensities_raw = (roi_mask.shape[0]) - df_pixels['y'].values

# 6. Clean and Interpolate
# Sort by wavenumber (since they are extracted in reverse order)
sort_idx = np.argsort(wavenumbers_raw)
w_sorted = wavenumbers_raw[sort_idx]
i_sorted = intensities_raw[sort_idx]

# Remove duplicates to avoid interpolation errors
_, unique_indices = np.unique(w_sorted, return_index=True)
w_unique = w_sorted[unique_indices]
i_unique = i_sorted[unique_indices]

# --- MODIFIED: Define standard range from 450 to 1800 ---
# Using np.arange(start, stop+1, step) for a clean 1 cm-1 resolution
standard_x = np.arange(450, 1801, 1)

# Use 'linear' or 'cubic' interpolation.
# Note: if your image doesn't reach 1800, 'extrapolate' will guess the values.
f = interp1d(w_unique, i_unique, kind='linear', bounds_error=False, fill_value="extrapolate")
interp_y = f(standard_x)

# 7. Smoothing & Normalization
y_smoothed = savgol_filter(interp_y, window_length=21, polyorder=3)
y_norm = (y_smoothed - np.min(y_smoothed)) / (np.max(y_smoothed) - np.min(y_smoothed))

# 8. Plot
fig = make_subplots(rows=1, cols=2, subplot_titles=("Original quercetin Image", "spectrum quercetin extraction"))
fig.add_trace(px.imshow(img).data[0], row=1, col=1)

fig.add_trace(go.Scatter(
    x=standard_x,
    y=y_norm,
    mode='lines',
    line=dict(color='blue', width=1.5),
    name='Intensity'
), row=1, col=2)

fig.update_layout(template='plotly_white', hovermode='x unified')
fig.show()

# 9. Save
df_final = pd.DataFrame({'Wavenumber': standard_x, 'Intensity': y_norm})
df_final.to_csv('quercetin_data.csv', index=False)

df_to_process = df_final # Assign df_final AFTER it has been created

peak_results = []

# Identify columns that contain spectral data (excluding 'Wavenumber')
intensity_cols = [col for col in df_to_process.columns if col != 'Wavenumber']

for col in intensity_cols:
    # Find peaks: adjust 'height' or 'prominence' based on your specific data noise levels
    # prominence=0.01 is a good starting point for normalized data
    peaks, _ = find_peaks(df_to_process[col], prominence=0.01)

    for peak_idx in peaks:
        peak_results.append({
            'Wavenumber': df_to_process['Wavenumber'].iloc[peak_idx],
            'Intensity': df_to_process[col].iloc[peak_idx]
        })

# Export peak position
# Create a DataFrame from the results
df_peaks = pd.DataFrame(peak_results)
df_peaks.to_csv('peak_positions_quercetin.csv', index=False)
print("Peak identification complete. Results saved to 'peak_positions_quercetin.csv'.")
df_peaks.head()

Peak identification complete. Results saved to 'peak_positions_quercetin.csv'.


,Wavenumber,Intensity
0,464,0.078039
1,497,0.085387
2,531,0.110309
3,611,0.182488
4,652,0.102310


# Rosmarinic acid

In [13]:
#------------ Raman spectrum of rosmarinic acid ----------------
#

# 1. Load Image
img_path = '/content/rosmarinic acid_re.png'
img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 2. Pre-process to Isolate the Line
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
# Using a slightly higher threshold to ensure the thin line is captured
_, threshold = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

# 3. Use Contour Detection to remove labels (numbers)
# This finds all "shapes" in the mask
contours, _ = cv2.findContours(threshold, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Create a blank mask to draw only the longest contour (the spectrum line)
clean_mask = np.zeros_like(threshold)
if contours:
    # The spectrum line is usually the contour with the largest area or length
    spectrum_contour = max(contours, key=cv2.contourArea)
    cv2.drawContours(clean_mask, [spectrum_contour], -1, 255, thickness=cv2.FILLED)

# 4. Extract Coordinates from the cleaned mask
y_coords, x_coords = np.where(clean_mask > 0)

if len(x_coords) == 0:
    print("Error: Detection failed. Trying fallback...")
    # Fallback: manually crop the area above the line to avoid labels
    # (Assuming labels are always placed vertically above peaks)
else:
    df_pixels = pd.DataFrame({'x': x_coords, 'y': y_coords})
    # Take the minimum Y (highest point) for each X to get the peak profile
    df_pixels = df_pixels.groupby('x').min().reset_index()

    x_pix = df_pixels['x'].values
    y_pix = df_pixels['y'].values

    # 5. Physical Axis Mapping (600 to 1800 cm⁻¹)
    x_min_pix, x_max_pix = x_pix.min(), x_pix.max()
    y_baseline_pix = np.max(y_coords)

    def pixel_to_unit(x, x_min, x_max, val_start, val_end):
        return val_start + (x - x_min) * (val_end - val_start) / (x_max - x_min)


    #but when running the code, the spectrum is expanded from original.
    wavenumbers_raw = pixel_to_unit(x_pix, x_min_pix, x_max_pix, 3420, 100)
    intensities_raw = (y_baseline_pix - y_pix)

    # 6. Standardization (Standard 1 cm⁻¹ step)
    standard_x = np.arange(450, 1801, 1)
    f = interp1d(wavenumbers_raw, intensities_raw, bounds_error=False, fill_value="extrapolate")
    interp_y = f(standard_x)

    # 7. Preprocessing
    # window_length must be odd. 11 is good for sharp peaks.
    y_smoothed = savgol_filter(interp_y, window_length=11, polyorder=3)

    # Baseline Removal
    base_obj = BaselineRemoval(y_smoothed)
    y_corrected = base_obj.IModPoly(degree=2)

    # Min-Max Normalization
    y_norm = (y_corrected - np.min(y_corrected)) / (np.max(y_corrected) - np.min(y_corrected))

    # 8. Interactive Comparison Plot
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Original rosmarinic acid Image", "rosmarinic acid spectrum extraction"))

    fig.add_trace(px.imshow(img_rgb).data[0], row=1, col=1)

    fig.add_trace(go.Scatter(
        x=standard_x,
        y=y_norm,
        mode='lines',
        line=dict(color='blue', width=1.5),
        name='Intensity'
    ), row=1, col=2)

    fig.update_layout(template='plotly_white', hovermode='x unified')
    fig.show()

    # 8. Export Data
    df_final = pd.DataFrame({'Wavenumber': standard_x, 'Intensity': y_norm})
    df_final.to_csv('rosmarinic acid_data.csv', index=False)
    print("Success: 'rosmarinic acid_data.csv' has been generated.")

    df_to_process = df_final # Assign df_final AFTER it has been created

peak_results = []

# Identify columns that contain spectral data (excluding 'Wavenumber')
intensity_cols = [col for col in df_to_process.columns if col != 'Wavenumber']

for col in intensity_cols:
    # Find peaks: adjust 'height' or 'prominence' based on your specific data noise levels
    # prominence=0.01 is a good starting point for normalized data
    peaks, _ = find_peaks(df_to_process[col], prominence=0.01)

    for peak_idx in peaks:
        peak_results.append({
            'Wavenumber': df_to_process['Wavenumber'].iloc[peak_idx],
            'Intensity': df_to_process[col].iloc[peak_idx]
        })

# Export peak position
# Create a DataFrame from the results
df_peaks = pd.DataFrame(peak_results)
df_peaks.to_csv('peak_positions_rosmarinic acid.csv', index=False)
print("Peak identification complete. Results saved to 'peak_positions_rosmarinic acid.csv'.")
df_peaks.head()



Success: 'rosmarinic acid_data.csv' has been generated.
Peak identification complete. Results saved to 'peak_positions_rosmarinic acid.csv'.


,Wavenumber,Intensity
0,603,0.013809
1,637,0.013513
2,747,0.033548
3,785,0.036774
4,813,0.033308


#------------ Raman spectrum of rosmarinic acid ----------------
#

# 1. Load Image
img_path = '/content/rosmarinic acid_re.png'
img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 2. Pre-process to Isolate the Line
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
# Using a slightly higher threshold to ensure the thin line is captured
_, threshold = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

# 3. Use Contour Detection to remove labels (numbers)
# This finds all "shapes" in the mask
contours, _ = cv2.findContours(threshold, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Create a blank mask to draw only the longest contour (the spectrum line)
clean_mask = np.zeros_like(threshold)
if contours:
    # The spectrum line is usually the contour with the largest area or length
    spectrum_contour = max(contours, key=cv2.contourArea)
    cv2.drawContours(clean_mask, [spectrum_contour], -1, 255, thickness=cv2.FILLED)

# 4. Extract Coordinates from the cleaned mask
y_coords, x_coords = np.where(clean_mask > 0)

if len(x_coords) == 0:
    print("Error: Detection failed. Trying fallback...")
    # Fallback: manually crop the area above the line to avoid labels
    # (Assuming labels are always placed vertically above peaks)
else:
    df_pixels = pd.DataFrame({'x': x_coords, 'y': y_coords})
    # Take the minimum Y (highest point) for each X to get the peak profile
    df_pixels = df_pixels.groupby('x').min().reset_index()

    x_pix = df_pixels['x'].values
    y_pix = df_pixels['y'].values

    # 5. Physical Axis Mapping (600 to 1800 cm⁻¹)
    x_min_pix, x_max_pix = x_pix.min(), x_pix.max()
    y_baseline_pix = np.max(y_coords)

    def pixel_to_unit(x, x_min, x_max, val_start, val_end):
        return val_start + (x - x_min) * (val_end - val_start) / (x_max - x_min)


    #but when running the code, the spectrum is expanded from original.
    wavenumbers_raw = pixel_to_unit(x_pix, x_min_pix, x_max_pix, 3420, 100)
    intensities_raw = (y_baseline_pix - y_pix)

    # 6. Standardization (Standard 1 cm⁻¹ step)
    standard_x = np.arange(450, 1801, 1)
    f = interp1d(wavenumbers_raw, intensities_raw, bounds_error=False, fill_value="extrapolate")
    interp_y = f(standard_x)

    # 7. Preprocessing
    # window_length must be odd. 11 is good for sharp peaks.
    y_smoothed = savgol_filter(interp_y, window_length=11, polyorder=3)

    # Baseline Removal
    base_obj = BaselineRemoval(y_smoothed)
    y_corrected = base_obj.IModPoly(degree=2)

    # Min-Max Normalization
    y_norm = (y_corrected - np.min(y_corrected)) / (np.max(y_corrected) - np.min(y_corrected))

    # 8. Interactive Comparison Plot
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Original rosmarinic acid Image", "rosmarinic acid spectrum extraction"))

    fig.add_trace(px.imshow(img_rgb).data[0], row=1, col=1)

    fig.add_trace(go.Scatter(
        x=standard_x,
        y=y_norm,
        mode='lines',
        line=dict(color='blue', width=1.5),
        name='Intensity'
    ), row=1, col=2)

    fig.update_layout(template='plotly_white', hovermode='x unified')
    fig.show()

    # 8. Export Data
    df_final = pd.DataFrame({'Wavenumber': standard_x, 'Intensity': y_norm})
    df_final.to_csv('rosmarinic acid_data.csv', index=False)
    print("Success: 'rosmarinic acid_data.csv' has been generated.")

    df_to_process = df_final # Assign df_final AFTER it has been created

peak_results = []

# Identify columns that contain spectral data (excluding 'Wavenumber')
intensity_cols = [col for col in df_to_process.columns if col != 'Wavenumber']

for col in intensity_cols:
    # Find peaks: adjust 'height' or 'prominence' based on your specific data noise levels
    # prominence=0.01 is a good starting point for normalized data
    peaks, _ = find_peaks(df_to_process[col], prominence=0.01)

    for peak_idx in peaks:
        peak_results.append({
            'Wavenumber': df_to_process['Wavenumber'].iloc[peak_idx],
            'Intensity': df_to_process[col].iloc[peak_idx]
        })

# Export peak position
# Create a DataFrame from the results
df_peaks = pd.DataFrame(peak_results)
df_peaks.to_csv('peak_positions_rosmarinic acid.csv', index=False)
print("Peak identification complete. Results saved to 'peak_positions_rosmarinic acid.csv'.")
df_peaks.head()



# Ginseng

In [14]:
# Three batches of Korean ginseng with specific different ginsenoside conformations were used: 248101 (4.24% w/v ginsenoside content), 246302 (4.81% w/v ginsenoside content) and 248102 (6.15% w/v ginsenoside content).

In [15]:
#------------ Raman spectrum of chinese ginseng ----------------
#result show closely the same to original

# 1. Load Image
img_path = '/content/chinese ginseng_re.png'
img = cv2.imread(img_path)
h, w, _ = img.shape

# 2. Define ROI (Region of Interest)
# This crops the image to ONLY the graph area, removing "b" and axis text
# Adjust these if your image size differs: [y1:y2, x1:x2]
roi = img[40:560, 80:1260]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)

# 3. Extract the Black Line
gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
# Using a tighter threshold to ignore gray grid lines
_, mask = cv2.threshold(gray, 130, 255, cv2.THRESH_BINARY_INV)

# 4. Extract Coordinates
y_coords, x_coords = np.where(mask > 0)

if len(x_coords) == 0:
    print("Error: No spectrum line detected inside ROI.")
else:
    # Clean noise: take the MINIMUM y (top of the black line) for every x pixel
    df_pixels = pd.DataFrame({'x': x_coords, 'y': y_coords})
    df_pixels = df_pixels.groupby('x').min().reset_index()

    x_pix = df_pixels['x'].values
    y_pix = df_pixels['y'].values

    # 5. Physical Axis Mapping (Calibrated)
    x_min_pix, x_max_pix = x_pix.min(), x_pix.max()
    y_baseline_pix = np.max(y_coords)

    def pixel_to_unit(x, x_min, x_max, val_start, val_end):
        return val_start + (x - x_min) * (val_end - val_start) / (x_max - x_min)


    #but when running the code, the spectrum is expanded from original.
    #we have randomly select the value to matching a peak, as much as possible
    wavenumbers_raw = pixel_to_unit(x_pix, x_min_pix, x_max_pix, 1680, 200)
    intensities_raw = (y_baseline_pix - y_pix)

    # 6. Standardization (200 - 1700 cm⁻¹)
    standard_x = np.arange(450, 1801, 1)

    # Sort raw data (interp1d requires ascending X)
    sort_idx = np.argsort(wavenumbers_raw)
    f = interp1d(wavenumbers_raw[sort_idx], intensities_raw[sort_idx],
                 bounds_error=False, fill_value="extrapolate")
    interp_y = f(standard_x)

    # 7. Preprocessing
    # Smoothing (window 11-15 is best for these sharp peaks)
    y_smoothed = savgol_filter(interp_y, window_length=11, polyorder=3)

    # Baseline Removal
    base_obj = BaselineRemoval(y_smoothed)
    y_corrected = base_obj.IModPoly(degree=2)

    # Min-Max Normalization
    y_norm = (y_corrected - np.min(y_corrected)) / (np.max(y_corrected) - np.min(y_corrected))

    # 8. Visualization
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Cropped ROI", "Corrected Extracted Spectrum"))

    fig.add_trace(go.Image(z=roi_rgb), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=standard_x, y=y_norm,
        mode='lines',
        line=dict(color='blue', width=1.5),
        name='Intensity'
    ), row=1, col=2)

    # Reverse the X-axis of the plot to match the original image visual (High to Low)
    fig.update_xaxes(title_text="Wavenumber (cm⁻¹)", row=1, col=2, autorange='reversed')
    fig.update_yaxes(title_text="Normalized Intensity", row=1, col=2)
    fig.update_layout(template='plotly_white', title_text="Ginseng Raman Data Extraction (Fixed)")
    fig.show()

    # 9. Export Data
    df_final = pd.DataFrame({'Wavenumber': standard_x, 'Intensity': y_norm})
    df_final.to_csv('chinese ginseng_data.csv', index=False)
    print("Success: Data saved to 'chinese ginseng_data.csv'.")

    df_to_process = df_final # Assign df_final AFTER it has been created

    peak_results = []

    # Identify columns that contain spectral data (excluding 'Wavenumber')
    intensity_cols = [col for col in df_to_process.columns if col != 'Wavenumber']

    for col in intensity_cols:
        # Find peaks: adjust 'height' or 'prominence' based on your specific data noise levels
        # prominence=0.01 is a good starting point for normalized data
        peaks, _ = find_peaks(df_to_process[col], prominence=0.01)

        for peak_idx in peaks:
            peak_results.append({
                'Wavenumber': df_to_process['Wavenumber'].iloc[peak_idx],
                'Intensity': df_to_process[col].iloc[peak_idx]
            })

    # Export peak position
    # Create a DataFrame from the results
    df_peaks = pd.DataFrame(peak_results)
    df_peaks.to_csv('peak_positions_chinese ginseng.csv', index=False)
    print("Peak identification complete. Results saved to 'peak_positions_chinese ginseng.csv'.")
    df_peaks.head()

Output hidden; open in https://colab.research.google.com to view.

In [16]:
#------------ Raman spectrum of Korean ginseng 248101 (4.24% w/v ginsenoside content)----------------
#result show closely the same to original

# 1. Load Image
img_path = '/content/Korean ginseng with 4.24% ginsenoside_re.png'
img = cv2.imread(img_path)
h, w, _ = img.shape

# 2. Define ROI (Region of Interest)
# This crops the image to ONLY the graph area, removing "b" and axis text
# Adjust these if your image size differs: [y1:y2, x1:x2]
roi = img[40:560, 80:1260]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)

# 3. Extract the Black Line
gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
# Using a tighter threshold to ignore gray grid lines
_, mask = cv2.threshold(gray, 130, 255, cv2.THRESH_BINARY_INV)

# 4. Extract Coordinates
y_coords, x_coords = np.where(mask > 0)

if len(x_coords) == 0:
    print("Error: No spectrum line detected inside ROI.")
else:
    # Clean noise: take the MINIMUM y (top of the black line) for every x pixel
    df_pixels = pd.DataFrame({'x': x_coords, 'y': y_coords})
    df_pixels = df_pixels.groupby('x').min().reset_index()

    x_pix = df_pixels['x'].values
    y_pix = df_pixels['y'].values

    # 5. Physical Axis Mapping (Calibrated)
    x_min_pix, x_max_pix = x_pix.min(), x_pix.max()
    y_baseline_pix = np.max(y_coords)

    def pixel_to_unit(x, x_min, x_max, val_start, val_end):
        return val_start + (x - x_min) * (val_end - val_start) / (x_max - x_min)


    #but when running the code, the spectrum is expanded from original.
    #we have randomly select the value to matching a peak, as much as possible
    wavenumbers_raw = pixel_to_unit(x_pix, x_min_pix, x_max_pix, 1620, 200)
    intensities_raw = (y_baseline_pix - y_pix)

    # 6. Standardization (200 - 1700 cm⁻¹)
    standard_x = np.arange(450, 1801, 1)

    # Sort raw data (interp1d requires ascending X)
    sort_idx = np.argsort(wavenumbers_raw)
    f = interp1d(wavenumbers_raw[sort_idx], intensities_raw[sort_idx],
                 bounds_error=False, fill_value="extrapolate")
    interp_y = f(standard_x)

    # 7. Preprocessing
    # Smoothing (window 11-15 is best for these sharp peaks)
    y_smoothed = savgol_filter(interp_y, window_length=11, polyorder=3)

    # Baseline Removal
    base_obj = BaselineRemoval(y_smoothed)
    y_corrected = base_obj.IModPoly(degree=2)

    # Min-Max Normalization
    y_norm = (y_corrected - np.min(y_corrected)) / (np.max(y_corrected) - np.min(y_corrected))

    # 8. Visualization
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Cropped ROI", "Corrected Extracted Spectrum"))

    fig.add_trace(go.Image(z=roi_rgb), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=standard_x, y=y_norm,
        mode='lines',
        line=dict(color='blue', width=1.5),
        name='Intensity'
    ), row=1, col=2)

    # Reverse the X-axis of the plot to match the original image visual (High to Low)
    fig.update_xaxes(title_text="Wavenumber (cm⁻¹)", row=1, col=2, autorange='reversed')
    fig.update_yaxes(title_text="Normalized Intensity", row=1, col=2)
    fig.update_layout(template='plotly_white', title_text="Ginseng Raman Data Extraction (Fixed)")
    fig.show()

    # 9. Export Data
    df_final = pd.DataFrame({'Wavenumber': standard_x, 'Intensity': y_norm})
    df_final.to_csv('Korean ginseng with 4.24% ginsenoside_data.csv', index=False)
    print("Success: Data saved to 'Korean ginseng with 4.24% ginsenoside_data.csv'.")

    df_to_process = df_final # Assign df_final AFTER it has been created

    peak_results = []

    # Identify columns that contain spectral data (excluding 'Wavenumber')
    intensity_cols = [col for col in df_to_process.columns if col != 'Wavenumber']

    for col in intensity_cols:
        # Find peaks: adjust 'height' or 'prominence' based on your specific data noise levels
        # prominence=0.01 is a good starting point for normalized data
        peaks, _ = find_peaks(df_to_process[col], prominence=0.01)

        for peak_idx in peaks:
            peak_results.append({
                'Wavenumber': df_to_process['Wavenumber'].iloc[peak_idx],
                'Intensity': df_to_process[col].iloc[peak_idx]
            })

    # Export peak position
    # Create a DataFrame from the results
    df_peaks = pd.DataFrame(peak_results)
    df_peaks.to_csv('peak_positions_Korean ginseng with 4.24% ginsenoside.csv', index=False)
    print("Peak identification complete. Results saved to 'peak_positions_Korean ginseng with 4.24% ginsenoside.csv'.")
    df_peaks.head()

Output hidden; open in https://colab.research.google.com to view.

In [17]:
#------------ Raman spectrum of Korean ginseng 248101 (4.81% w/v ginsenoside content)----------------
#result show closely the same to original

# 1. Load Image
img_path = '/content/Korean ginseng with 4.81% ginsenoside_re.png'
img = cv2.imread(img_path)
h, w, _ = img.shape

# 2. Define ROI (Region of Interest)
# This crops the image to ONLY the graph area, removing "b" and axis text
# Adjust these if your image size differs: [y1:y2, x1:x2]
roi = img[40:560, 80:1260]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)

# 3. Extract the Black Line
gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
# Using a tighter threshold to ignore gray grid lines
_, mask = cv2.threshold(gray, 130, 255, cv2.THRESH_BINARY_INV)

# 4. Extract Coordinates
y_coords, x_coords = np.where(mask > 0)

if len(x_coords) == 0:
    print("Error: No spectrum line detected inside ROI.")
else:
    # Clean noise: take the MINIMUM y (top of the black line) for every x pixel
    df_pixels = pd.DataFrame({'x': x_coords, 'y': y_coords})
    df_pixels = df_pixels.groupby('x').min().reset_index()

    x_pix = df_pixels['x'].values
    y_pix = df_pixels['y'].values

    # 5. Physical Axis Mapping (Calibrated)
    x_min_pix, x_max_pix = x_pix.min(), x_pix.max()
    y_baseline_pix = np.max(y_coords)

    def pixel_to_unit(x, x_min, x_max, val_start, val_end):
        return val_start + (x - x_min) * (val_end - val_start) / (x_max - x_min)


    #but when running the code, the spectrum is expanded from original.
    #we have randomly select the value to matching a peak, as much as possible
    wavenumbers_raw = pixel_to_unit(x_pix, x_min_pix, x_max_pix, 1680, 230)
    intensities_raw = (y_baseline_pix - y_pix)

    # 6. Standardization (200 - 1700 cm⁻¹)
    standard_x = np.arange(450, 1801, 1)

    # Sort raw data (interp1d requires ascending X)
    sort_idx = np.argsort(wavenumbers_raw)
    f = interp1d(wavenumbers_raw[sort_idx], intensities_raw[sort_idx],
                 bounds_error=False, fill_value="extrapolate")
    interp_y = f(standard_x)

    # 7. Preprocessing
    # Smoothing (window 11-15 is best for these sharp peaks)
    y_smoothed = savgol_filter(interp_y, window_length=11, polyorder=3)

    # Baseline Removal
    base_obj = BaselineRemoval(y_smoothed)
    y_corrected = base_obj.IModPoly(degree=2)

    # Min-Max Normalization
    y_norm = (y_corrected - np.min(y_corrected)) / (np.max(y_corrected) - np.min(y_corrected))

    # 8. Visualization
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Cropped ROI", "Corrected Extracted Spectrum"))

    fig.add_trace(go.Image(z=roi_rgb), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=standard_x, y=y_norm,
        mode='lines',
        line=dict(color='blue', width=1.5),
        name='Intensity'
    ), row=1, col=2)

    # Reverse the X-axis of the plot to match the original image visual (High to Low)
    fig.update_xaxes(title_text="Wavenumber (cm⁻¹)", row=1, col=2, autorange='reversed')
    fig.update_yaxes(title_text="Normalized Intensity", row=1, col=2)
    fig.update_layout(template='plotly_white', title_text="Ginseng Raman Data Extraction (Fixed)")
    fig.show()

    # 9. Export Data
    df_final = pd.DataFrame({'Wavenumber': standard_x, 'Intensity': y_norm})
    df_final.to_csv('Korean ginseng with 4.81% ginsenoside_data.csv', index=False)
    print("Success: Data saved to 'Korean ginseng with 4.81% ginsenoside_data.csv'.")

    df_to_process = df_final # Assign df_final AFTER it has been created

    peak_results = []

    # Identify columns that contain spectral data (excluding 'Wavenumber')
    intensity_cols = [col for col in df_to_process.columns if col != 'Wavenumber']

    for col in intensity_cols:
        # Find peaks: adjust 'height' or 'prominence' based on your specific data noise levels
        # prominence=0.01 is a good starting point for normalized data
        peaks, _ = find_peaks(df_to_process[col], prominence=0.01)

        for peak_idx in peaks:
            peak_results.append({
                'Wavenumber': df_to_process['Wavenumber'].iloc[peak_idx],
                'Intensity': df_to_process[col].iloc[peak_idx]
            })

    # Export peak position
    # Create a DataFrame from the results
    df_peaks = pd.DataFrame(peak_results)
    df_peaks.to_csv('peak_positions_Korean ginseng with 4.81% ginsenoside.csv', index=False)
    print("Peak identification complete. Results saved to 'peak_positions_Korean ginseng with 4.81% ginsenoside.csv'.")
    df_peaks.head()

Output hidden; open in https://colab.research.google.com to view.

In [18]:
#------------ Raman spectrum of Korean ginseng 248102 (6.15% w/v ginsenoside content)----------------
#result show closely the same to original

# 1. Load Image
img_path = '/content/Korean ginseng with 6.15% ginsenoside_re.png'
img = cv2.imread(img_path)
h, w, _ = img.shape

# 2. Define ROI (Region of Interest)
# This crops the image to ONLY the graph area, removing "b" and axis text
# Adjust these if your image size differs: [y1:y2, x1:x2]
roi = img[40:560, 80:1260]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)

# 3. Extract the Black Line
gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
# Using a tighter threshold to ignore gray grid lines
_, mask = cv2.threshold(gray, 130, 255, cv2.THRESH_BINARY_INV)

# 4. Extract Coordinates
y_coords, x_coords = np.where(mask > 0)

if len(x_coords) == 0:
    print("Error: No spectrum line detected inside ROI.")
else:
    # Clean noise: take the MINIMUM y (top of the black line) for every x pixel
    df_pixels = pd.DataFrame({'x': x_coords, 'y': y_coords})
    df_pixels = df_pixels.groupby('x').min().reset_index()

    x_pix = df_pixels['x'].values
    y_pix = df_pixels['y'].values

    # 5. Physical Axis Mapping (Calibrated)
    x_min_pix, x_max_pix = x_pix.min(), x_pix.max()
    y_baseline_pix = np.max(y_coords)

    def pixel_to_unit(x, x_min, x_max, val_start, val_end):
        return val_start + (x - x_min) * (val_end - val_start) / (x_max - x_min)


    #but when running the code, the spectrum is expanded from original.
    #we have randomly select the value to matching a peak, as much as possible
    wavenumbers_raw = pixel_to_unit(x_pix, x_min_pix, x_max_pix, 1650, 205)
    intensities_raw = (y_baseline_pix - y_pix)

    # 6. Standardization (200 - 1700 cm⁻¹)
    standard_x = np.arange(450, 1801, 1)

    # Sort raw data (interp1d requires ascending X)
    sort_idx = np.argsort(wavenumbers_raw)
    f = interp1d(wavenumbers_raw[sort_idx], intensities_raw[sort_idx],
                 bounds_error=False, fill_value="extrapolate")
    interp_y = f(standard_x)

    # 7. Preprocessing
    # Smoothing (window 11-15 is best for these sharp peaks)
    y_smoothed = savgol_filter(interp_y, window_length=11, polyorder=3)

    # Baseline Removal
    base_obj = BaselineRemoval(y_smoothed)
    y_corrected = base_obj.IModPoly(degree=2)

    # Min-Max Normalization
    y_norm = (y_corrected - np.min(y_corrected)) / (np.max(y_corrected) - np.min(y_corrected))

    # 8. Visualization
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Cropped ROI", "Corrected Extracted Spectrum"))

    fig.add_trace(go.Image(z=roi_rgb), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=standard_x, y=y_norm,
        mode='lines',
        line=dict(color='blue', width=1.5),
        name='Intensity'
    ), row=1, col=2)

    # Reverse the X-axis of the plot to match the original image visual (High to Low)
    fig.update_xaxes(title_text="Wavenumber (cm⁻¹)", row=1, col=2, autorange='reversed')
    fig.update_yaxes(title_text="Normalized Intensity", row=1, col=2)
    fig.update_layout(template='plotly_white', title_text="Ginseng Raman Data Extraction (Fixed)")
    fig.show()

    # 9. Export Data
    df_final = pd.DataFrame({'Wavenumber': standard_x, 'Intensity': y_norm})
    df_final.to_csv('Korean ginseng with 6.15% ginsenoside_data.csv', index=False)
    print("Success: Data saved to 'Korean ginseng with 6.15% ginsenoside_data.csv'.")

    df_to_process = df_final # Assign df_final AFTER it has been created

    peak_results = []

    # Identify columns that contain spectral data (excluding 'Wavenumber')
    intensity_cols = [col for col in df_to_process.columns if col != 'Wavenumber']

    for col in intensity_cols:
        # Find peaks: adjust 'height' or 'prominence' based on your specific data noise levels
        # prominence=0.01 is a good starting point for normalized data
        peaks, _ = find_peaks(df_to_process[col], prominence=0.01)

        for peak_idx in peaks:
            peak_results.append({
                'Wavenumber': df_to_process['Wavenumber'].iloc[peak_idx],
                'Intensity': df_to_process[col].iloc[peak_idx]
            })

    # Export peak position
    # Create a DataFrame from the results
    df_peaks = pd.DataFrame(peak_results)
    df_peaks.to_csv('peak_positions_Korean ginseng with 6.15% ginsenoside.csv', index=False)
    print("Peak identification complete. Results saved to 'peak_positions_Korean ginseng with 6.15% ginsenoside.csv'.")
    df_peaks.head()

Output hidden; open in https://colab.research.google.com to view.